# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VinayBhavikatti/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I rank content pages for review using two simple signals: staleness and search visibility.

Pages that are older and still have meaningful search impressions receive a higher score and are placed earlier in the review queue.

Reason codes:
- `STALE_VISIBLE` — old page with meaningful search visibility.
- `STALE` — old page with lower visibility.
- `VISIBLE` — meaningful visibility but not stale.
- `LOW_PRIORITY` — neither signal is strong.

The rule uses only information available at the decision time and does not use future outcomes or label-derived fields.

In [1]:
# Section 1 — Rule definition

# The rule uses only decision-time signals.
# Higher score = higher priority for review.

STALE_DAYS = 180
VISIBLE_IMPRESSIONS = 100

print("Baseline rule:")
print(f"- Stale: days_since_last_update >= {STALE_DAYS}")
print(f"- Visible: impressions_90d >= {VISIBLE_IMPRESSIONS}")
print()
print("Reason codes:")
print("STALE_VISIBLE  = stale + visible")
print("STALE          = stale only")
print("VISIBLE        = visible only")
print("LOW_PRIORITY   = neither")


Baseline rule:
- Stale: days_since_last_update >= 180
- Visible: impressions_90d >= 100

Reason codes:
STALE_VISIBLE  = stale + visible
STALE          = stale only
VISIBLE        = visible only
LOW_PRIORITY   = neither


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# Section 2 — Build the ranked queue

import os
import pandas as pd

# Clone repo if it is not already present
if not os.path.exists("/content/flyrank-ml-internship"):
    !git clone https://github.com/VinayBhavikatti/flyrank-ml-internship.git

# Move into repo
%cd /content/flyrank-ml-internship

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create baseline score
# Higher score = higher review priority
df["baseline_score"] = (
    (df["content_age_days"] >= 180).astype(int) * 2
    + (df["impressions_90d"] >= 100).astype(int)
)

# Reason code
df["reason_code"] = "LOW_PRIORITY"

df.loc[
    (df["content_age_days"] >= 180) &
    (df["impressions_90d"] >= 100),
    "reason_code"
] = "STALE_VISIBLE"

df.loc[
    (df["content_age_days"] >= 180) &
    (df["impressions_90d"] < 100),
    "reason_code"
] = "STALE"

df.loc[
    (df["content_age_days"] < 180) &
    (df["impressions_90d"] >= 100),
    "reason_code"
] = "VISIBLE"

# Action label
df["action"] = "MONITOR"

df.loc[df["baseline_score"] >= 2, "action"] = "REVIEW"
df.loc[df["baseline_score"] == 1, "action"] = "CHECK"

# Rank the queue
queue = df.sort_values(
    ["baseline_score", "impressions_90d", "content_age_days"],
    ascending=[False, False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# Save output
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows ranked:", len(queue))

print("\nTop 20:")
display(
    queue[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "content_age_days",
            "impressions_90d"
        ]
    ].head(20)
)

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 141 (delta 51), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.89 MiB | 9.66 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/flyrank-ml-internship
Saved: work/outputs/baseline_action_score.csv
Rows ranked: 30000

Top 20:


,rank,content_id,baseline_score,reason_code,action,content_age_days,impressions_90d
0,1,content_5fe46e04994d,3,STALE_VISIBLE,REVIEW,537,517715
1,2,content_aaef01a50def,3,STALE_VISIBLE,REVIEW,445,517109
2,3,content_8c19996aa890,3,STALE_VISIBLE,REVIEW,445,509252
3,4,content_4c36c775b818,3,STALE_VISIBLE,REVIEW,445,463103
4,5,content_2dba2b1f9536,3,STALE_VISIBLE,REVIEW,299,443434
5,6,content_1a9e894be2e2,3,STALE_VISIBLE,REVIEW,482,416180
6,7,content_2c2606c5d176,3,STALE_VISIBLE,REVIEW,362,347399
7,8,content_db5989a78dd3,3,STALE_VISIBLE,REVIEW,445,345111
8,9,content_9532f197bbc8,3,STALE_VISIBLE,REVIEW,445,309192
9,10,content_8e7ba84a972b,3,STALE_VISIBLE,REVIEW,224,288426


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Top-20 review
top20 = queue.head(20).copy()

# Add a simple confidence note and what could make the recommendation wrong
top20["confidence_note"] = "High - both age and visibility signals are strong"

top20["what_would_make_it_wrong"] = (
    "Recent update, unusual traffic change, or data-quality issue could make this review priority misleading."
)

# Show the review table
review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "baseline_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)


,rank,content_id,action,reason_code,baseline_score,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
1,2,content_aaef01a50def,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
2,3,content_8c19996aa890,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
3,4,content_4c36c775b818,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
4,5,content_2dba2b1f9536,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
5,6,content_1a9e894be2e2,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
6,7,content_2c2606c5d176,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
7,8,content_db5989a78dd3,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
8,9,content_9532f197bbc8,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."
9,10,content_8e7ba84a972b,REVIEW,STALE_VISIBLE,3,High - both age and visibility signals are strong,"Recent update, unusual traffic change, or data..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 4. Weak picks
# Show the lowest-priority rows so we can check whether the rule makes sensible decisions.

weak_picks = queue.tail(10).copy()

weak_picks["review_note"] = (
    "Weak pick: low baseline priority because the page does not have strong "
    "staleness and visibility signals."
)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "content_age_days",
            "impressions_90d",
            "review_note"
        ]
    ]
)

,rank,content_id,baseline_score,reason_code,action,content_age_days,impressions_90d,review_note
29990,29991,content_1d9eca1ce9cd,0,LOW_PRIORITY,MONITOR,91,1,Weak pick: low baseline priority because the p...
29991,29992,content_7e1125021dce,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...
29992,29993,content_f9047f009a60,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...
29993,29994,content_fb301c227850,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...
29994,29995,content_b045313ddc67,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...
29995,29996,content_3913b54f1805,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...
29996,29997,content_6959fda268ea,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...
29997,29998,content_74b78c2e3b4b,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...
29998,29999,content_b813d68af8df,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...
29999,30000,content_aa0c15fa00d5,0,LOW_PRIORITY,MONITOR,90,1,Weak pick: low baseline priority because the p...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.